# RAG Enrichment Evaluation

**Notebook:** 02 — Verify that `retrieve_context()` returns plausible, relevant results for different anomaly types.

This notebook loads sample alerts from `data/alerts.jsonl`, calls `retrieve_context()` from `src.rag`, and manually inspects whether each retrieval makes sense for the given anomaly type.

In [1]:
import json
import sys
from pathlib import Path

# Ensure project root is on sys.path so src.rag can be imported
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.rag import retrieve_context

print("Imports OK")

Imports OK


In [2]:
ALERTS_PATH = REPO_ROOT / "data" / "alerts.jsonl"
alerts = []
with open(ALERTS_PATH) as f:
    for line in f:
        alerts.append(json.loads(line))

print(f"Loaded {len(alerts)} alerts from {ALERTS_PATH}")

# Pick one alert per anomaly type based on window_summary patterns
scanning_alert = None
cavitation_alert = None
physics_alert = None

for a in alerts:
    s = a["window_summary"]
    fcs = s["function_codes"]
    ips = s["source_ips"]
    has_write = s["has_write_fc"]
    tank_max = s["tank_level_max"]
    tank_mean = s["tank_level_mean"]

    # Scanning: has_write_fc=1 (FC 6 observed) and non-HMI IP
    if scanning_alert is None and has_write == 1 and "172.24.0.10" in ips:
        scanning_alert = a
    # Cavitation: tank_mean below 50 (dropping fast)
    if cavitation_alert is None and tank_mean < 20:
        cavitation_alert = a
    # Physics violation: high tank + valve open (unusual combination)
    if physics_alert is None and tank_max > 80 and 6 not in fcs:
        physics_alert = a

print(f"Scanning alert:     {scanning_alert is not None}")
print(f"Cavitation alert:   {cavitation_alert is not None}")
print(f"Physics alert:      {physics_alert is not None}")

Loaded 69 alerts from /media/liamcarvajal/Storage/code/ics-agentic-soc-pipeline/data/alerts.jsonl
Scanning alert:     True
Cavitation alert:   True
Physics alert:      True


## Retrieval Test 1: Scanning + Unauthorized Write

This alert contains FC 6 (write), FC 3 (read), two source IPs (172.22.0.10 and 172.24.0.10), and tank level 61.8-80.4. The query should match the unauthorized write incident and the IEC 62443 controls about least privilege and Modbus filtering.

In [3]:
if scanning_alert:
    print("Alert summary:")
    print(f"  PLC: {scanning_alert['plc_ip']}")
    print(f"  FCs: {scanning_alert['window_summary']['function_codes']}")
    print(f"  Source IPs: {scanning_alert['window_summary']['source_ips']}")
    print(f"  Has write FC: {scanning_alert['window_summary']['has_write_fc']}")
    print()

    ctx = retrieve_context(scanning_alert)
    print(ctx)

Alert summary:
  PLC: 172.21.0.10
  FCs: [3, 6]
  Source IPs: ['172.24.0.10', '172.22.0.10']
  Has write FC: 1



/media/liamcarvajal/Storage/code/ics-agentic-soc-pipeline/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6210.93it/s]

--- ASSET INFO ---

## eng-ws-01 (172.23.0.4)

The Engineering Workstation is the sole authorized source for Modbus write commands (FC 6, FC 16) to the PLCs. It is located in a physically secured area with access control. Under normal operation it writes only during maintenance windows or process setpoint changes. Any write from an IP other than this workstation is considered an unauthorized write anomaly.

## plc-intake (172.21.0.10)

The intake PLC controls the water intake process in the OT-Security-Lab. It manages the inlet valve (reg_0) and reports tank level (reg_5) for the raw water holding tank. This is a Critical asset in the Level 1 Control Zone running OpenPLC v4 firmware 4.0.7. It communicates via Modbus TCP on port 502. The HMI (172.22.0.10) polls it every 5 seconds for register data, and the engineering workstation (172.23.0.4) is the only authorized source for Modbus write commands (FC 6, FC 16).

--- IEC 62443 CONTROLS ---

## Least Privilege and Role-Based Access Contr

## Retrieval Test 2: Cavitation

This alert has a very low tank level (mean < 20%), indicating the tank is nearly empty. The query should retrieve the cavitation incident and IEC 62443 controls about anomaly detection.

In [4]:
if cavitation_alert:
    print("Alert summary:")
    print(f"  PLC: {cavitation_alert['plc_ip']}")
    print(f"  Tank mean: {cavitation_alert['window_summary']['tank_level_mean']:.1f}%")
    print(f"  Tank max: {cavitation_alert['window_summary']['tank_level_max']:.1f}%")
    print()

    ctx = retrieve_context(cavitation_alert)
    print(ctx)

Alert summary:
  PLC: 172.21.0.10
  Tank mean: 19.5%
  Tank max: 21.2%

--- ASSET INFO ---

## plc-distribution (172.21.0.12)

The distribution PLC manages the water tower and distribution pumps. Like plc-treatment, it is a sibling PLC contacted only by the HMI under normal conditions. It runs the same OpenPLC v4 firmware. Anomalous traffic to this asset from unrecognized IPs indicates lateral movement or network reconnaissance.

## plc-intake (172.21.0.10)

The intake PLC controls the water intake process in the OT-Security-Lab. It manages the inlet valve (reg_0) and reports tank level (reg_5) for the raw water holding tank. This is a Critical asset in the Level 1 Control Zone running OpenPLC v4 firmware 4.0.7. It communicates via Modbus TCP on port 502. The HMI (172.22.0.10) polls it every 5 seconds for register data, and the engineering workstation (172.23.0.4) is the only authorized source for Modbus write commands (FC 6, FC 16).

--- IEC 62443 CONTROLS ---

## Network Segmentation

## Retrieval Test 3: Physics Violation

This alert has high tank levels with anomalous valve behavior. The query should retrieve asset info and applicable controls.

In [5]:
if physics_alert:
    print("Alert summary:")
    print(f"  PLC: {physics_alert['plc_ip']}")
    print(f"  Tank mean: {physics_alert['window_summary']['tank_level_mean']:.1f}%")
    print(f"  Tank max: {physics_alert['window_summary']['tank_level_max']:.1f}%")
    print(f"  FCs: {physics_alert['window_summary']['function_codes']}")
    print()

    ctx = retrieve_context(physics_alert)
    print(ctx)

Alert summary:
  PLC: 172.21.0.10
  Tank mean: 25.9%
  Tank max: 92.2%
  FCs: [3, 131]



--- ASSET INFO ---

## plc-intake (172.21.0.10)

The intake PLC controls the water intake process in the OT-Security-Lab. It manages the inlet valve (reg_0) and reports tank level (reg_5) for the raw water holding tank. This is a Critical asset in the Level 1 Control Zone running OpenPLC v4 firmware 4.0.7. It communicates via Modbus TCP on port 502. The HMI (172.22.0.10) polls it every 5 seconds for register data, and the engineering workstation (172.23.0.4) is the only authorized source for Modbus write commands (FC 6, FC 16).

## plc-distribution (172.21.0.12)

The distribution PLC manages the water tower and distribution pumps. Like plc-treatment, it is a sibling PLC contacted only by the HMI under normal conditions. It runs the same OpenPLC v4 firmware. Anomalous traffic to this asset from unrecognized IPs indicates lateral movement or network reconnaissance.

--- IEC 62443 CONTROLS ---

## Network Segmentation (SR 5.1)

The OT network is divided into security zones: Level 0 (Proce

## Manual Analysis

### Test 1: Scanning + Unauthorized Write (172.24.0.10, FC 3+6)
- **Asset info:** Does retrieval mention plc-intake (172.21.0.10)?  Does it correctly identify 172.24.0.10 as unauthorized?
- **Controls:** Does the context mention least privilege (SR 1.4) or Modbus filtering (SR 4.1)?
- **Past incidents:** Does the 2024-11-12 unauthorized write incident appear?

### Test 2: Cavitation
- **Asset info:** Does the asset info mention tank level behavior or cavitation risk?
- **Controls:** Does the context include anomaly detection controls?
- **Past incidents:** Does the 2024-11-12 incident match (it involved cavitation)?

### Test 3: Physics Violation
- **Asset info:** Does the retrieval correctly identify the affected PLC?
- **Controls:** Are the relevant IEC 62443 controls returned?

### Overall Assessment
- Is the context specific enough to help an analyst or LLM agent?
- Are there any irrelevant chunks mixed in?
- How could the query or chunking be improved?